# 04 — GAM Modeling
## Generalized Additive Model: PM2.5 ~ Weather Variables

A GAM fits a smooth, nonlinear function for each predictor while keeping the
model interpretable via partial effect plots. This is appropriate because
aerosol–weather relationships are unlikely to be strictly linear.

**Model:** `PM2.5_LRAPA ~ s(temperature) + s(humidity) + s(wind_speed) + s(wind_dir_sin) + s(wind_dir_cos) + s(pressure) + s(hour)`

**Library:** `pygam` — install with `pip install pygam`

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from sklearn.metrics import r2_score, mean_squared_error
from pygam import LinearGAM, s, f

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

## 0. Load & Prepare Data

In [ ]:
df = pd.read_csv('../data/processed/analysis_data.csv', parse_dates=['timestamp'])
print(f'Loaded {len(df):,} records')

# Features used in the GAM
FEATURES = [
    'temperature_f',
    'humidity',
    'wind_speed_mph',
    'wind_dir_sin',
    'wind_dir_cos',
    'pressure_hpa',
    'hour',
]
TARGET = 'pm2.5_lrapa'

# Drop rows with any missing values in features or target
FEATURES = [f for f in FEATURES if f in df.columns]
# Keep timestamp for chronological train/test split (dropped after sort)
model_df = df[['timestamp', TARGET] + FEATURES].dropna().copy()
print(f'Model dataset: {len(model_df):,} complete rows')
print(f'Features: {FEATURES}')
model_df[FEATURES + [TARGET]].describe().round(2)

## 1. Train/Test Split

Using a temporal split: first 80% of timestamps for training, last 20% for testing.
This avoids data leakage that would occur with random splitting of time series data.

In [ ]:
model_df = model_df.sort_values('timestamp').reset_index(drop=True)
model_df = model_df.drop(columns=['timestamp'])

split_idx = int(len(model_df) * 0.8)

train = model_df.iloc[:split_idx]
test  = model_df.iloc[split_idx:]

X_train = train[FEATURES].values
y_train = train[TARGET].values
X_test  = test[FEATURES].values
y_test  = test[TARGET].values

print(f'Train: {len(train):,} records')
print(f'Test:  {len(test):,} records')

## 2. Fit GAM

`LinearGAM` fits a penalized regression spline for each predictor.
The penalty (lambda) controls smoothness; `pygam` can tune it automatically with `gridsearch`.

In [ ]:
# Build the GAM formula: one smooth term per feature
terms = s(0)
for i in range(1, len(FEATURES)):
    terms = terms + s(i)

gam = LinearGAM(terms)

# Tune smoothing parameters via grid search (may take ~30s)
print('Fitting GAM with grid-search smoothing parameter tuning...')
gam.gridsearch(X_train, y_train, progress=True)
print('Done.')

In [ ]:
# Model summary
print(gam.summary())

# Performance metrics
y_pred_train = gam.predict(X_train)
y_pred_test  = gam.predict(X_test)

train_r2   = r2_score(y_train, y_pred_train)
test_r2    = r2_score(y_test,  y_pred_test)
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
test_rmse  = np.sqrt(mean_squared_error(y_test,  y_pred_test))

print(f'\nTrain  R²: {train_r2:.3f}   RMSE: {train_rmse:.2f} µg/m³')
print(f'Test   R²: {test_r2:.3f}   RMSE: {test_rmse:.2f} µg/m³')

## 3. Partial Effect Plots

Each panel shows the fitted smooth function for one predictor, holding all others at their mean. The shaded band is a 95% confidence interval.

In [ ]:
n_features = len(FEATURES)
n_cols = 4
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = axes.flatten()

for i, (ax, fname) in enumerate(zip(axes, FEATURES)):
    XX = gam.generate_X_grid(term=i)
    pdep, confi = gam.partial_dependence(term=i, X=XX, width=0.95)

    ax.fill_between(XX[:, i], confi[:, 0], confi[:, 1], alpha=0.25, color='steelblue')
    ax.plot(XX[:, i], pdep, color='steelblue', linewidth=2)
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.set_xlabel(fname)
    ax.set_ylabel('Partial effect (µg/m³)')
    ax.set_title(fname)

# Hide unused axes
for ax in axes[n_features:]:
    ax.set_visible(False)

plt.suptitle('GAM Partial Effects — PM2.5 ~ Weather Variables', y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig('../data/processed/fig_gam_partial_effects.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Model Diagnostics

In [ ]:
residuals = y_test - y_pred_test

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Predicted vs actual
ax = axes[0]
ax.scatter(y_test, y_pred_test, s=6, alpha=0.4, color='steelblue')
lim = max(y_test.max(), y_pred_test.max())
ax.plot([0, lim], [0, lim], 'k--', linewidth=1)
ax.set_xlabel('Actual PM2.5 (µg/m³)')
ax.set_ylabel('Predicted PM2.5 (µg/m³)')
ax.set_title(f'Predicted vs Actual\nTest R² = {test_r2:.3f}')

# Residuals vs predicted
ax = axes[1]
ax.scatter(y_pred_test, residuals, s=6, alpha=0.4, color='tomato')
ax.axhline(0, color='black', linewidth=1)
ax.set_xlabel('Predicted PM2.5 (µg/m³)')
ax.set_ylabel('Residual (µg/m³)')
ax.set_title('Residuals vs Predicted')

# QQ plot of residuals
ax = axes[2]
stats.probplot(residuals, plot=ax)
ax.set_title('Q-Q Plot of Residuals')

plt.tight_layout()
plt.savefig('../data/processed/fig_gam_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Results Summary

In [ ]:
print('=' * 55)
print('GAM Model Results Summary')
print('=' * 55)
print(f'Response variable:   {TARGET}')
print(f'Predictor variables: {FEATURES}')
print(f'Training records:    {len(train):,}')
print(f'Test records:        {len(test):,}')
print()
print(f'Train R²:   {train_r2:.3f}')
print(f'Test  R²:   {test_r2:.3f}')
print(f'Train RMSE: {train_rmse:.2f} µg/m³')
print(f'Test  RMSE: {test_rmse:.2f} µg/m³')
print()
print('Smoothing parameters (lambda) per term:')
for fname, lam in zip(FEATURES, gam.lam):
    print(f'  {fname:20s}: {float(lam):.4f}')